> **Chapter Map:** This notebook is the companion for **Chapter 2** (Computational Perspectives on Surface Catalysis). It contains three parts with suggested session breaks. Parts II and III build on Part I concepts but can be started independently.

# NB2: Computational Perspectives on Surface Catalysis

## Learning Objectives

After completing this notebook, you will be able to:

**Part I -- Adsorption Fundamentals**
1. Implement and visualize the Langmuir isotherm equation
2. Create coverage heatmaps showing combined effects of temperature and pressure
3. Model competitive adsorption and catalyst poisoning
4. Analyze TPD spectra using the Redhead equation to extract desorption energies

**Part II -- Surface Kinetics**
5. Implement Langmuir-Hinshelwood (LH), Eley-Rideal (ER), and Mars-van Krevelen (MVK) rate expressions
6. Identify characteristic "fingerprints" of each mechanism from rate-vs-pressure plots
7. Fit power-law and mechanistic models to experimental data using `scipy.optimize.curve_fit`
8. Use residual analysis to discriminate between candidate models

**Part III -- Temperature Dependence**
9. Perform Arrhenius analysis to extract $E_a$ and pre-exponential factor $A$ with uncertainties
10. Perform Eyring analysis to extract $\Delta H^\ddagger$ and $\Delta S^\ddagger$
11. Explain how surface coverage affects the apparent activation energy
12. Recognize and interpret the enthalpy-entropy compensation effect

## Background

This notebook develops the computational tools for the three pillars of surface catalysis:

- **Adsorption** (Part I): How molecules bind to surfaces -- the Langmuir isotherm and its extensions
- **Kinetics** (Part II): How surface reactions proceed -- LH, ER, and MVK mechanisms
- **Temperature dependence** (Part III): How reaction rates change with temperature -- Arrhenius and Eyring analysis

Together, these tools form a complete workflow: measure rates at multiple temperatures and pressures, identify the mechanism from fingerprint diagnostics, and extract activation parameters.

## Prerequisites

- **Concepts:** Surface coverage, adsorption equilibrium, rate laws (Chapter 2)
- **Python skills:** numpy arrays, matplotlib, curve fitting (NB1)
- **Previous notebooks:** NB1 (Bridge: CSV, plotting, linear fitting)

---
## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit, fsolve
from scipy.integrate import solve_ivp
from scipy import stats

# Set default plot style for publication-quality figures
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Colorblind-safe color palette (Wong, 2011)
COLORS = {
    'blue': '#0072B2',
    'orange': '#E69F00',
    'green': '#009E73',
    'vermillion': '#D55E00',
    'skyblue': '#56B4E9',
    'purple': '#CC79A7'
}

# Physical constants
R = 8.314          # Gas constant, J/(mol K)
kB = 1.380649e-23  # Boltzmann constant, J/K
h_planck = 6.62607e-34    # Planck constant, J s
ln_kB_over_h = np.log(kB / h_planck)  # = 23.76

print("Setup complete. Libraries imported successfully.")
print(f"ln(kB/h) = {ln_kB_over_h:.2f}")

---
# Part I: Adsorption Fundamentals

Adsorption is the fundamental first step in heterogeneous catalysis. Before a molecule can react on a catalyst surface, it must first adsorb. The **Langmuir isotherm** provides a simple yet powerful model for understanding how surface coverage depends on gas-phase pressure and temperature.

### Key Equations

**Langmuir Isotherm:**
$$\theta = \frac{KP}{1 + KP}$$

where $\theta$ is the fractional surface coverage, $K$ is the adsorption equilibrium constant (bar$^{-1}$), and $P$ is the partial pressure (bar).

**Temperature Dependence of K (van 't Hoff):**
$$K(T) = K_0 \exp\left(-\frac{\Delta H_{ads}}{RT}\right)$$

**Competitive Adsorption:**
$$\theta_A = \frac{K_A P_A}{1 + K_A P_A + K_I P_I}$$

## 1.1 Langmuir Isotherm

We begin by implementing the Langmuir isotherm and exploring how the equilibrium constant $K$ affects surface coverage.

In [ ]:
def langmuir_coverage(P, K):
    """
    Calculate fractional surface coverage using the Langmuir isotherm.

    Parameters
    ----------
    P : float or array-like
        Partial pressure of the adsorbate (bar)
    K : float
        Adsorption equilibrium constant (bar^-1)

    Returns
    -------
    float or ndarray
        Fractional surface coverage theta (dimensionless, 0 to 1)

    Notes
    -----
    Assumes monolayer adsorption, equivalent sites, no lateral interactions.
    """
    P = np.asarray(P)
    return K * P / (1 + K * P)


# Test: at P = 1/K, we expect theta = 0.5
K_test = 1.0  # bar^-1
P_half = 1 / K_test
theta_half = langmuir_coverage(P_half, K_test)
print(f"Test: At P = 1/K = {P_half:.2f} bar, theta = {theta_half:.3f}")
print(f"Expected: theta = 0.500")

### Plotting Langmuir Isotherms for Different K Values

The equilibrium constant $K$ determines how strongly a molecule binds to the surface:
- **Large K**: Strong adsorption, surface saturates at low pressure
- **Small K**: Weak adsorption, requires high pressure for significant coverage

In [ ]:
# Define pressure range (logarithmic for wide coverage)
P_values = np.logspace(-3, 2, 500)  # 0.001 to 100 bar

# ADJUSTABLE PARAMETERS
K_values = [0.1, 1, 10]  # bar^-1
colors_K = [COLORS['blue'], COLORS['green'], COLORS['vermillion']]
linestyles_K = ['-', '--', '-.']

fig, ax = plt.subplots(figsize=(10, 7))

for K, color, ls in zip(K_values, colors_K, linestyles_K):
    theta = langmuir_coverage(P_values, K)
    ax.semilogx(P_values, theta, color=color, linestyle=ls,
                label=f'K = {K} bar$^{{-1}}$')

    # Mark half-coverage point (P = 1/K)
    P_half = 1 / K
    ax.plot(P_half, 0.5, 'o', color=color, markersize=10,
            markeredgecolor='black', markeredgewidth=1.5)
    ax.annotate(f'$P_{{1/2}}$ = {P_half:.1f} bar',
                xy=(P_half, 0.5),
                xytext=(P_half * 3, 0.5 + 0.08),
                fontsize=10, color=color,
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

ax.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, alpha=0.7)

# Annotate regimes
ax.annotate("Henry's Law\nRegime\n($\\theta \\approx KP$)",
            xy=(0.005, 0.15), fontsize=11, ha='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.annotate('Saturation\nRegime\n($\\theta \\approx 1$)',
            xy=(30, 0.85), fontsize=11, ha='center',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))

ax.set_xlabel('Pressure, $P$ (bar)')
ax.set_ylabel('Surface Coverage, $\\theta$ (dimensionless)')
ax.set_title('Langmuir Isotherms: Effect of Equilibrium Constant $K$')
ax.set_xlim(1e-3, 100)
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

### Key Observations

1. **Half-coverage pressure**: $P_{1/2} = 1/K$ -- the pressure at which $\theta = 0.5$
2. **Henry's law regime** (low $P$): $\theta \approx KP$ (linear dependence)
3. **Saturation regime** (high $P$): $\theta \approx 1$ (all sites occupied)
4. Larger $K$ means stronger adsorption and lower $P_{1/2}$

**Concept Check:** Before looking at the coverage map: if temperature *increases* at constant pressure, will coverage increase or decrease for *exothermic* adsorption ($\Delta H_\text{ads} < 0$)? Write your prediction, then run the next cell to check.

---
## 1.2 Coverage Map: Combined Temperature and Pressure Effects

In real catalytic processes, both temperature and pressure vary. Since adsorption is typically exothermic ($\Delta H_{ads} < 0$), higher temperatures reduce $K$ and thus reduce coverage.

In [ ]:
def K_temperature(T, K0, delta_H):
    """
    Temperature-dependent adsorption equilibrium constant (van 't Hoff).

    Parameters
    ----------
    T : float or array-like
        Temperature (K)
    K0 : float
        Pre-exponential factor (bar^-1)
    delta_H : float
        Enthalpy of adsorption (J/mol), negative for exothermic

    Returns
    -------
    float or ndarray
        Equilibrium constant K at temperature T (bar^-1)
    """
    T = np.asarray(T)
    return K0 * np.exp(-delta_H / (R * T))

In [ ]:
# ADJUSTABLE PARAMETERS
K0 = 1e-3          # Pre-exponential factor (bar^-1)
delta_H = -60e3    # Enthalpy of adsorption (J/mol)

# Create temperature and pressure grids
T_range = np.linspace(300, 700, 100)
P_range = np.logspace(-3, 1, 100)
T_grid, P_grid = np.meshgrid(T_range, P_range)

# Calculate coverage for each (T, P) combination
K_grid = K_temperature(T_grid, K0, delta_H)
theta_grid = langmuir_coverage(P_grid, K_grid)

# Create the heatmap
fig, ax = plt.subplots(figsize=(10, 8))
c = ax.pcolormesh(T_range, P_range, theta_grid,
                  cmap='viridis', shading='auto', vmin=0, vmax=1)
cbar = fig.colorbar(c, ax=ax, label='Surface Coverage, $\\theta$')
ax.set_yscale('log')

# Add contour lines
contour_levels = [0.1, 0.3, 0.5, 0.7, 0.9]
CS = ax.contour(T_range, P_range, theta_grid, levels=contour_levels,
                colors='white', linewidths=1.5, linestyles='--')
ax.clabel(CS, inline=True, fontsize=10, fmt='$\\theta$ = %.1f')

ax.set_xlabel('Temperature (K)')
ax.set_ylabel('Pressure (bar)')
ax.set_title(f'Coverage Map: $\\theta(T, P)$\n'
             f'$K_0$ = {K0:.0e} bar$^{{-1}}$, '
             f'$\\Delta H_{{ads}}$ = {delta_H/1000:.0f} kJ/mol')
plt.tight_layout()
plt.show()

print(f"K at 300 K: {K_temperature(300, K0, delta_H):.2f} bar^-1")
print(f"K at 500 K: {K_temperature(500, K0, delta_H):.4f} bar^-1")
print(f"K at 700 K: {K_temperature(700, K0, delta_H):.6f} bar^-1")

### Interpretation of the Coverage Map

1. **High coverage (yellow)**: Low temperature + High pressure
2. **Low coverage (dark purple)**: High temperature + Low pressure
3. **Iso-coverage contours**: Show combinations of $(T, P)$ that give the same coverage

**Practical implications:**
- To maintain constant coverage as $T$ increases, $P$ must also increase
- At very high $T$, even high $P$ cannot achieve full surface coverage
- The $\theta = 0.5$ contour is particularly useful for catalyst design

---
## 1.3 Competitive Adsorption and Catalyst Poisoning

In real catalytic systems, multiple species compete for adsorption sites. Poisons (or inhibitors) adsorb strongly but do not participate productively in the reaction, blocking sites from the reactant.

In [ ]:
def competitive_langmuir(P_A, K_A, P_I, K_I):
    """
    Coverage of species A in the presence of inhibitor I.

    Parameters
    ----------
    P_A : float or array-like
        Partial pressure of species A (bar)
    K_A : float
        Adsorption equilibrium constant for A (bar^-1)
    P_I : float or array-like
        Partial pressure of inhibitor I (bar)
    K_I : float
        Adsorption equilibrium constant for inhibitor I (bar^-1)

    Returns
    -------
    float or ndarray
        Fractional surface coverage of A (dimensionless)
    """
    P_A = np.asarray(P_A)
    P_I = np.asarray(P_I)
    return K_A * P_A / (1 + K_A * P_A + K_I * P_I)

In [ ]:
# ADJUSTABLE PARAMETERS
K_A = 2.0       # Reactant equilibrium constant (bar^-1)
P_A = 1.0       # Reactant partial pressure (bar)
K_I_values = [10, 100, 1000]  # Inhibitor equilibrium constants (bar^-1)
P_I_range = np.logspace(-4, 0, 200)

# Baseline coverage (no inhibitor)
theta_A_pure = langmuir_coverage(P_A, K_A)

fig, ax = plt.subplots(figsize=(10, 7))
colors_poison = [COLORS['blue'], COLORS['green'], COLORS['vermillion']]
linestyles_poison = ['-', '--', '-.']

ax.axhline(y=theta_A_pure, color='black', linestyle=':', linewidth=2,
           label=f'No inhibitor ($\\theta_A$ = {theta_A_pure:.2f})')

for K_I, color, ls in zip(K_I_values, colors_poison, linestyles_poison):
    theta_A = competitive_langmuir(P_A, K_A, P_I_range, K_I)
    ax.semilogx(P_I_range, theta_A, color=color, linestyle=ls,
                linewidth=2.5,
                label=f'$K_I$ = {K_I} bar$^{{-1}}$')

ax.axhline(y=theta_A_pure / 2, color='gray', linestyle='--',
           linewidth=1, alpha=0.5)
ax.text(1.2e-4, theta_A_pure / 2 + 0.02, '50% reduction',
        fontsize=10, color='gray')

ax.set_xlabel('Inhibitor Partial Pressure, $P_I$ (bar)')
ax.set_ylabel('Reactant Coverage, $\\theta_A$ (dimensionless)')
ax.set_title(f'Catalyst Poisoning: Effect of Inhibitor\n'
             f'$K_A$ = {K_A} bar$^{{-1}}$, $P_A$ = {P_A} bar')
ax.set_xlim(1e-4, 1)
ax.set_ylim(0, 0.75)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

### Interpretation of Poisoning Curves

1. **Stronger poisons** (larger $K_I$) cause significant coverage loss at lower concentrations
2. The ratio $K_I/K_A$ determines poisoning severity
3. Sulfur compounds (H$_2$S, SO$_2$) are notorious catalyst poisons with very large $K_I$
4. Some "poisons" at low concentrations can be beneficial (selectivity modifiers)

---
## 1.4 Temperature-Programmed Desorption (TPD) Analysis

TPD is a key experimental technique for characterizing adsorbate-surface interactions. The sample is heated at a constant rate $\beta$ (K/min), and the desorption rate is recorded as a function of temperature.

The **Redhead equation** provides a quick estimate of desorption activation energy from the peak temperature:

$$E_d = RT_p\left[\ln\left(\frac{\nu T_p}{\beta}\right) - 3.64\right]$$

**Rule of thumb:** $E_d$ (kJ/mol) $\approx 0.28 \times T_p$ (K) for $\beta = 10$ K/min.

In [ ]:
def asymmetric_gaussian(T, T_center, sigma, amplitude,
                        asymmetry=0.3, asym_width=20.0):
    """
    Generate an asymmetric Gaussian peak typical of TPD spectra.

    Parameters
    ----------
    T : array-like
        Temperature array (K)
    T_center : float
        Approximate center of the peak (K)
    sigma : float
        Peak width / standard deviation (K)
    amplitude : float
        Peak amplitude (proportional to initial coverage)
    asymmetry : float, optional
        Asymmetry factor (0 = symmetric). Default 0.3.
    asym_width : float, optional
        Width of the asymmetry transition (K). Default 20.0.

    Returns
    -------
    ndarray
        Desorption rate signal (arbitrary units)
    """
    T = np.asarray(T)
    gaussian = np.exp(-((T - T_center) ** 2) / (2 * sigma ** 2))
    asymmetry_factor = 1 + asymmetry * np.tanh(-(T - T_center) / asym_width)
    return amplitude * gaussian * asymmetry_factor


def redhead_equation(T_peak, nu=1e13, beta=10.0):
    """
    Desorption activation energy from the Redhead equation.

    Parameters
    ----------
    T_peak : float
        Temperature at peak maximum (K)
    nu : float, optional
        Pre-exponential factor (s^-1). Default 1e13.
    beta : float, optional
        Heating rate (K/s). Default 10.0 K/s.

    Returns
    -------
    float
        Desorption activation energy (J/mol)

    Notes
    -----
    Rule of thumb: E_d (kJ/mol) ~ 0.25 * T_p (K) for beta = 10 K/s,
    or ~ 0.28 * T_p (K) for beta = 10 K/min.
    """
    E_d = R * T_peak * (np.log(nu * T_peak / beta) - 3.64)
    return E_d


def find_peak_maximum(T, signal):
    """Find temperature and intensity at the peak maximum."""
    idx_max = np.argmax(signal)
    return T[idx_max], signal[idx_max]


# Test the Redhead equation
T_p_test = 400  # K
beta_test = 10 / 60  # 10 K/min -> K/s
E_d_test = redhead_equation(T_p_test, nu=1e13, beta=beta_test)
print(f"Redhead Test: T_p = {T_p_test} K, beta = 10 K/min")
print(f"  E_d = {E_d_test/1000:.1f} kJ/mol")
print(f"  Rule of thumb (0.28 x T_p): {0.28 * T_p_test:.0f} kJ/mol")

In [ ]:
# ADJUSTABLE PARAMETERS
T_min, T_max, n_points = 300, 700, 500

# Weak binding site (alpha state)
T_center_1 = 400; sigma_1 = 25; asym_1 = 0.35; width_1 = 20; A_1 = 0.5
# Strong binding site (beta state)
T_center_2 = 520; sigma_2 = 30; asym_2 = 0.30; width_2 = 25; A_2 = 1.0

# Experimental parameters
nu = 1e13          # Pre-exponential factor (s^-1)
beta = 10.0 / 60   # Heating rate: 10 K/min -> K/s

# Generate TPD spectrum
T_tpd = np.linspace(T_min, T_max, n_points)
signal_weak = asymmetric_gaussian(T_tpd, T_center_1, sigma_1, A_1,
                                  asym_1, width_1)
signal_strong = asymmetric_gaussian(T_tpd, T_center_2, sigma_2, A_2,
                                    asym_2, width_2)
signal_total = signal_weak + signal_strong

# Find peaks and calculate desorption energies
T_p_weak, h_weak = find_peak_maximum(T_tpd, signal_weak)
T_p_strong, h_strong = find_peak_maximum(T_tpd, signal_strong)
E_d_weak = redhead_equation(T_p_weak, nu=nu, beta=beta)
E_d_strong = redhead_equation(T_p_strong, nu=nu, beta=beta)

# Plot
fig, ax = plt.subplots(figsize=(11, 7))
ax.plot(T_tpd, signal_weak, '--', color=COLORS['orange'], linewidth=2.5,
        label=f'Weak site (alpha): $E_d$ = {E_d_weak/1000:.1f} kJ/mol')
ax.plot(T_tpd, signal_strong, '-', color=COLORS['blue'], linewidth=2.5,
        label=f'Strong site (beta): $E_d$ = {E_d_strong/1000:.1f} kJ/mol')
ax.plot(T_tpd, signal_total, '-', color='gray', linewidth=2, alpha=0.5,
        label='Total spectrum')

ax.axvline(x=T_p_weak, color=COLORS['orange'], linestyle=':', alpha=0.7)
ax.axvline(x=T_p_strong, color=COLORS['blue'], linestyle=':', alpha=0.7)

ax.annotate(f'$T_p^\\alpha$ = {T_p_weak:.0f} K',
            xy=(T_p_weak, h_weak),
            xytext=(T_p_weak - 30, h_weak + 0.1),
            fontsize=11, color=COLORS['orange'],
            arrowprops=dict(arrowstyle='->', color=COLORS['orange']))
ax.annotate(f'$T_p^\\beta$ = {T_p_strong:.0f} K',
            xy=(T_p_strong, h_strong),
            xytext=(T_p_strong + 20, h_strong + 0.1),
            fontsize=11, color=COLORS['blue'],
            arrowprops=dict(arrowstyle='->', color=COLORS['blue']))

props = dict(boxstyle='round', facecolor='white', edgecolor='gray',
             alpha=0.9)
textbox = (f"Redhead Analysis\n"
           f"nu = {nu:.0e} s^-1\n"
           f"beta = {beta*60:.0f} K/min\n"
           f"$E_d = RT_p[\\ln(\\nu T_p/\\beta) - 3.64]$")
ax.text(0.02, 0.98, textbox, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', fontfamily='monospace', bbox=props)

ax.set_xlabel('Temperature (K)')
ax.set_ylabel('Desorption Rate (a.u.)')
ax.set_title('Temperature-Programmed Desorption (TPD) Spectrum')
ax.set_xlim(T_min, T_max)
ax.set_ylim(0, None)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Validity check
ratio_w = E_d_weak / (R * T_p_weak)
ratio_s = E_d_strong / (R * T_p_strong)
print(f"\nRedhead Validity (E_d/RT_p should be 20-50):")
print(f"  Weak site:   {ratio_w:.1f}")
print(f"  Strong site: {ratio_s:.1f}")

### TPD Key Observations

1. **Peak position implies binding energy**: Higher $T_p$ indicates stronger adsorbate-surface binding
2. **Asymmetric peak shape**: TPD peaks have a sharp leading edge and gradual trailing edge
3. **Multiple peaks**: Different site types produce distinct peaks
4. **First-order desorption**: $T_p$ is independent of initial coverage
5. **Second-order desorption** (recombinative): $T_p$ shifts to lower $T$ with higher coverage

### Key Takeaways — Part I: Adsorption Fundamentals

- The Langmuir isotherm $\theta = KP/(1+KP)$ is the foundation; competitive adsorption adds inhibitor terms to the denominator
- Coverage maps $\theta(T,P)$ reveal operating windows where adsorption is neither too weak nor saturated  
- TPD peak temperature relates to desorption energy via the Redhead equation
- The van 't Hoff equation connects K(T) to the heat of adsorption

> **Session Break (suggested):** Save your work. Parts II and III build on Part I concepts but can be started fresh.

---
# Part II: Surface Kinetics -- Mechanism Fingerprints

In heterogeneous catalysis, the observed reaction rate depends on the mechanism by which reactants interact with the catalyst surface. Three fundamental mechanisms are:

| Mechanism | Key Feature | Rate Maximum in $P_A$? |
|-----------|-------------|----------------------|
| Langmuir-Hinshelwood (LH) | Both reactants adsorb | **Yes** (site blocking) |
| Eley-Rideal (ER) | Only one reactant adsorbs | No (monotonic saturation) |
| Mars-van Krevelen (MVK) | Catalyst redox cycle | No (two-resistance saturation) |

Each mechanism produces a characteristic rate-vs-pressure "fingerprint" that can be used for model discrimination.

## 2.1 Langmuir-Hinshelwood Rate Law

Both reactants adsorb before reacting on the surface:

1. $A_{(g)} + * \rightleftharpoons A^* \quad (K_A)$
2. $B_{(g)} + * \rightleftharpoons B^* \quad (K_B)$
3. $A^* + B^* \longrightarrow \text{Products} + 2*$ (rate-determining step)

$$r = k \frac{K_A P_A \cdot K_B P_B}{(1 + K_A P_A + K_B P_B)^2}$$

The squared denominator produces the characteristic **rise-maximum-fall** behavior. The maximum occurs at $P_{A,\text{max}} = (1 + K_B P_B) / K_A$.

In [ ]:
def lh_rate_bimolecular(P_A, P_B, k, K_A, K_B):
    """
    Bimolecular Langmuir-Hinshelwood reaction rate.

    Parameters
    ----------
    P_A, P_B : float or array-like
        Partial pressures of reactants A and B (bar)
    k : float
        Surface reaction rate constant (mol/(m^2 s))
    K_A, K_B : float
        Adsorption equilibrium constants (bar^-1)

    Returns
    -------
    float or ndarray
        Reaction rate in mol/(m^2 s)
    """
    P_A = np.asarray(P_A)
    P_B = np.asarray(P_B)
    denom = (1 + K_A * P_A + K_B * P_B)**2
    return k * K_A * P_A * K_B * P_B / denom


# Test
r_test = lh_rate_bimolecular(P_A=0.6, P_B=0.2, k=1, K_A=2, K_B=1)
print(f"Test: r(P_A=0.6, P_B=0.2) = {r_test:.4f}")
print(f"Maximum at P_A = (1 + K_B*P_B)/K_A = "
      f"{(1 + 1*0.2)/2:.2f} bar")

In [ ]:
# ADJUSTABLE PARAMETERS
k_lh = 1.0; K_A_lh = 2.0; K_B_lh = 1.0; P_B_lh = 0.2
P_A_range = np.linspace(0.01, 5.0, 500)

rate_lh = lh_rate_bimolecular(P_A_range, P_B_lh, k_lh, K_A_lh, K_B_lh)
P_A_max = (1 + K_B_lh * P_B_lh) / K_A_lh
r_max = lh_rate_bimolecular(P_A_max, P_B_lh, k_lh, K_A_lh, K_B_lh)

fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(P_A_range, rate_lh, color=COLORS['blue'], linewidth=2.5,
        label='LH bimolecular')
ax.plot(P_A_max, r_max, 'o', color=COLORS['vermillion'], markersize=12,
        markeredgecolor='black', markeredgewidth=1.5, zorder=5)
ax.annotate(f'Maximum at $P_A$ = {P_A_max:.2f} bar',
            xy=(P_A_max, r_max),
            xytext=(P_A_max + 0.8, r_max + 0.005),
            fontsize=11, color=COLORS['vermillion'],
            arrowprops=dict(arrowstyle='->',
                            color=COLORS['vermillion'], lw=1.5))

ax.annotate('First order in $A$\n(low coverage)',
            xy=(0.2, 0.02), fontsize=10, ha='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.annotate('Inverse order in $A$\n(site blocking)',
            xy=(3.5, 0.015), fontsize=10, ha='center',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))

ax.set_xlabel('Partial Pressure $P_A$ (bar)')
ax.set_ylabel('Reaction Rate, $r$ (mol m$^{-2}$ s$^{-1}$)')
ax.set_title('Langmuir-Hinshelwood: Rise-Maximum-Fall Behavior\n'
             f'$k$ = {k_lh}, $K_A$ = {K_A_lh} bar$^{{-1}}$, '
             f'$K_B$ = {K_B_lh} bar$^{{-1}}$, $P_B$ = {P_B_lh} bar')
ax.legend(loc='upper right')
ax.set_xlim(0, 5); ax.set_ylim(0, None)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Concept Check:** The LH bimolecular rate has a maximum at an intermediate $P_A$. Without calculating, predict: does the $P_A$ at which the maximum occurs shift left or right if $K_A$ increases? Modify `K_A` above and re-run to verify.

### Effect of $K_A$ on the Rate Maximum

Stronger adsorption of A (larger $K_A$) shifts the rate maximum to lower $P_A$ and makes self-inhibition more pronounced.

In [ ]:
K_A_values = [0.5, 2, 8]
colors_ka = [COLORS['green'], COLORS['blue'], COLORS['vermillion']]
linestyles_ka = ['-', '--', '-.']

fig, ax = plt.subplots(figsize=(10, 7))
for K_A_val, color, ls in zip(K_A_values, colors_ka, linestyles_ka):
    rate = lh_rate_bimolecular(P_A_range, P_B_lh, k_lh, K_A_val, K_B_lh)
    P_max = (1 + K_B_lh * P_B_lh) / K_A_val
    r_at_max = lh_rate_bimolecular(P_max, P_B_lh, k_lh, K_A_val, K_B_lh)
    ax.plot(P_A_range, rate, color=color, linestyle=ls, linewidth=2.5,
            label=f'$K_A$ = {K_A_val} bar$^{{-1}}$ '
                  f'(max at {P_max:.2f} bar)')
    ax.plot(P_max, r_at_max, 'o', color=color, markersize=10,
            markeredgecolor='black', markeredgewidth=1.5, zorder=5)

ax.set_xlabel('Partial Pressure $P_A$ (bar)')
ax.set_ylabel('Reaction Rate, $r$ (mol m$^{-2}$ s$^{-1}$)')
ax.set_title('Effect of $K_A$ on LH Rate Maximum')
ax.legend(loc='upper right')
ax.set_xlim(0, 5); ax.set_ylim(0, None)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2.2 Eley-Rideal Rate Law

Only species A adsorbs; B reacts directly from the gas phase:

$$r = k_{ER} \cdot P_B \cdot \frac{K_A P_A}{1 + K_A P_A}$$

Critical difference from LH: **no rate maximum in $P_A$**. Since B does not compete for sites, increasing $P_A$ always increases (or saturates) the rate.

In [ ]:
def er_rate(P_A, P_B, k_er, K_A):
    """
    Eley-Rideal reaction rate.

    Parameters
    ----------
    P_A : float or array-like
        Partial pressure of the adsorbing species A (bar)
    P_B : float or array-like
        Partial pressure of the gas-phase species B (bar)
    k_er : float
        ER rate constant (mol m^-2 s^-1 bar^-1)
    K_A : float
        Adsorption equilibrium constant for A (bar^-1)

    Returns
    -------
    float or ndarray
        Reaction rate in mol/(m^2 s)
    """
    P_A = np.asarray(P_A)
    P_B = np.asarray(P_B)
    theta_A = K_A * P_A / (1 + K_A * P_A)
    return k_er * P_B * theta_A


# Test monotonic behavior
r_low = er_rate(0.1, 0.5, 1.0, 2.0)
r_high = er_rate(10.0, 0.5, 1.0, 2.0)
print(f"ER rate at P_A = 0.1: {r_low:.4f}")
print(f"ER rate at P_A = 10.0: {r_high:.4f}")
print(f"Monotonically increasing: {r_high > r_low}")

---
## 2.3 Mars-van Krevelen (MVK) Mechanism

The catalyst donates **lattice oxygen** through a redox cycle:

$$r = \frac{k_{red} \cdot k_{ox} \cdot P_A \cdot P_{O_2}}{k_{red} \cdot P_A + k_{ox} \cdot P_{O_2}}$$

This has the form of "two resistances in series" -- the overall rate is limited by whichever half-cycle is slower.

In [ ]:
def mvk_rate(P_A, P_O2, k_ox, k_red):
    """
    Mars-van Krevelen reaction rate.

    Parameters
    ----------
    P_A : float or array-like
        Partial pressure of the substrate (bar)
    P_O2 : float or array-like
        Partial pressure of O2 (bar)
    k_ox : float
        Oxidation rate constant — rate of substrate oxidation by lattice O
        (mol m^-2 s^-1 bar^-1)
    k_red : float
        Re-oxidation rate constant — rate of lattice re-oxidation by O2
        (mol m^-2 s^-1 bar^-1)

    Returns
    -------
    float or ndarray
        Reaction rate in mol/(m^2 s)

    Notes
    -----
    Matches the signature in utils.py: mvk_rate(P_A, P_O2, k_ox, k_red).
    """
    P_A = np.asarray(P_A)
    P_O2 = np.asarray(P_O2)
    return k_ox * k_red * P_A * P_O2 / (k_ox * P_A + k_red * P_O2)


# Limiting behavior test
print(f"O2-rich (P_A=0.01): r = {mvk_rate(0.01, 1.0, 0.5, 1.0):.4f}, "
      f"expected ~ k_ox*P_A = {0.5*0.01:.4f}")
print(f"O2-lean (P_A=100): r = {mvk_rate(100, 1.0, 0.5, 1.0):.4f}, "
      f"expected ~ k_red*P_O2 = {1.0*1.0:.4f}")

---
## 2.4 Mechanism Fingerprints -- Diagnostic Comparison

| Feature | LH | ER | MVK |
|---------|----|----|-----|
| Both reactants adsorb | Yes | No (one only) | Redox cycle |
| Rate maximum in $P_A$ | **Yes** | No | No |
| Inverse order possible | Yes | No | No |
| Denominator exponent | 2 | 1 | 1 (sum form) |
| High-$P_A$ limit | $r \propto 1/P_A$ | $r \to$ const | $r \to$ const |

In [ ]:
# Three-panel fingerprint comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
P_A_fp = np.linspace(0.01, 5.0, 500)

# LH
rate_fp_lh = lh_rate_bimolecular(P_A_fp, 0.5, 1.0, 2.0, 1.0)
axes[0].plot(P_A_fp, rate_fp_lh, color=COLORS['blue'], linewidth=2.5)
axes[0].set_title('Langmuir-Hinshelwood', fontsize=13, fontweight='bold')
axes[0].text(0.5, 0.85, 'Rise-Maximum-Fall',
             transform=axes[0].transAxes, fontsize=10,
             ha='center', style='italic', color=COLORS['blue'])

# ER
rate_fp_er = er_rate(P_A_fp, 0.5, 1.0, 2.0)
axes[1].plot(P_A_fp, rate_fp_er, color=COLORS['orange'], linewidth=2.5)
axes[1].set_title('Eley-Rideal', fontsize=13, fontweight='bold')
axes[1].text(0.5, 0.85, 'Monotonic Saturation',
             transform=axes[1].transAxes, fontsize=10,
             ha='center', style='italic', color=COLORS['orange'])

# MVK
rate_fp_mvk = mvk_rate(P_A_fp, 0.5, 0.5, 1.0)
axes[2].plot(P_A_fp, rate_fp_mvk, color=COLORS['green'], linewidth=2.5)
axes[2].set_title('Mars-van Krevelen', fontsize=13, fontweight='bold')
axes[2].text(0.5, 0.85, 'Two-Resistance Saturation',
             transform=axes[2].transAxes, fontsize=10,
             ha='center', style='italic', color=COLORS['green'])

for ax in axes:
    ax.set_xlabel('$P_A$ (bar)')
    ax.set_ylabel('Rate, $r$')
    ax.set_xlim(0, 5); ax.set_ylim(0, None)
    ax.grid(True, alpha=0.3)

plt.suptitle('Mechanism Fingerprints: $r$ vs $P_A$ at Fixed $P_B$',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Diagnostic summary:")
print("  1. Does r(P_A) show a MAXIMUM? --> LH")
print("  2. Monotonic saturation? --> ER or MVK")
print("  3. To distinguish ER from MVK: vary P_B/P_O2 independently")

---
## 2.5 Model Discrimination with Experimental Data

The workflow for mechanism identification:
1. Load rate data at a single temperature
2. Fit candidate models (power-law, LH, ER) using `curve_fit`
3. Compare fit quality using $R^2$ **and** residual analysis
4. Check that fitted parameters are physically reasonable

In [ ]:
# Load kinetic data
df = pd.read_csv('data/kinetic_data_sample.csv', comment='#')
mask_400K = df['temperature_K'] == 400
P_CO_data = df.loc[mask_400K, 'P_CO_Pa'].values
rate_data = df.loc[mask_400K, 'rate_mol_m2_s'].values
rate_unc = df.loc[mask_400K, 'rate_uncertainty'].values
P_CO_bar = P_CO_data / 1e5  # Convert Pa to bar

print("Data at T = 400 K:")
print("-" * 50)
for P, r, u in zip(P_CO_bar, rate_data, rate_unc):
    print(f"  P_CO = {P:.4f} bar, r = {r:.2e} +/- {u:.1e}")

In [ ]:
def power_law(P, k_app, n):
    """Power-law rate: r = k_app * P^n."""
    return k_app * np.asarray(P)**n


def lh_model_1var(P_A, a, b):
    """
    Simplified LH model at fixed P_B: r = a*P_A / (1 + b*P_A)^2.

    Rate maximum occurs at P_A = 1/b.
    """
    P_A = np.asarray(P_A)
    return a * P_A / (1 + b * P_A)**2


# Fit power-law
popt_pl, pcov_pl = curve_fit(power_law, P_CO_bar, rate_data,
                              p0=[1e-4, 0.5],
                              sigma=rate_unc, absolute_sigma=True)
rate_pred_pl = power_law(P_CO_bar, *popt_pl)
ss_res_pl = np.sum((rate_data - rate_pred_pl)**2)
ss_tot = np.sum((rate_data - np.mean(rate_data))**2)
r2_pl = 1 - ss_res_pl / ss_tot

# Fit LH model
popt_lh, pcov_lh = curve_fit(lh_model_1var, P_CO_bar, rate_data,
                              p0=[1e-4, 50],
                              sigma=rate_unc, absolute_sigma=True)
rate_pred_lh = lh_model_1var(P_CO_bar, *popt_lh)
ss_res_lh = np.sum((rate_data - rate_pred_lh)**2)
r2_lh = 1 - ss_res_lh / ss_tot

print("Power-Law Fit:")
print(f"  k_app = {popt_pl[0]:.2e}, n = {popt_pl[1]:.3f}, "
      f"R^2 = {r2_pl:.6f}")
print(f"\nLH Model Fit:")
print(f"  a = {popt_lh[0]:.2e}, b = {popt_lh[1]:.1f} bar^-1, "
      f"R^2 = {r2_lh:.6f}")
print(f"  Rate maximum at P_CO = 1/b = {1/popt_lh[1]:.4f} bar")

In [ ]:
# Comparison plot with residuals
P_smooth = np.linspace(P_CO_bar.min() * 0.5,
                        P_CO_bar.max() * 1.5, 200)

fig, ((ax1), (ax2)) = plt.subplots(1, 2, figsize=(14, 6))

# Left: Data + fits
ax1.errorbar(P_CO_bar, rate_data, yerr=rate_unc,
             fmt='o', markersize=10,
             markerfacecolor=COLORS['blue'],
             markeredgecolor='black', markeredgewidth=1.5,
             ecolor='gray', capsize=5, label='Data', zorder=5)
ax1.plot(P_smooth, power_law(P_smooth, *popt_pl), '--',
         color=COLORS['orange'], linewidth=2,
         label=f'Power law ($R^2$ = {r2_pl:.4f})')
ax1.plot(P_smooth, lh_model_1var(P_smooth, *popt_lh), '-',
         color=COLORS['vermillion'], linewidth=2,
         label=f'LH model ($R^2$ = {r2_lh:.4f})')
ax1.set_xlabel('$P_{CO}$ (bar)')
ax1.set_ylabel('Rate (mol m$^{-2}$ s$^{-1}$)')
ax1.set_title('Model Comparison (T = 400 K)')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# Right: Residuals
norm_resid_pl = (rate_data - rate_pred_pl) / rate_unc
norm_resid_lh = (rate_data - rate_pred_lh) / rate_unc
ax2.plot(P_CO_bar, norm_resid_pl, 's', markersize=10,
         markerfacecolor=COLORS['orange'],
         markeredgecolor='black', markeredgewidth=1.5,
         label='Power law')
ax2.plot(P_CO_bar, norm_resid_lh, 'o', markersize=10,
         markerfacecolor=COLORS['vermillion'],
         markeredgecolor='black', markeredgewidth=1.5,
         label='LH model')
ax2.axhline(y=0, color='black', linewidth=1)
ax2.axhline(y=2, color='gray', linestyle='--', alpha=0.5)
ax2.axhline(y=-2, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('$P_{CO}$ (bar)')
ax2.set_ylabel('Normalized Residual')
ax2.set_title('Residual Analysis')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Residual analysis reveals systematic patterns in the power-law")
print("fit that R^2 alone cannot detect.")

### Key Takeaways — Part II: Surface Kinetics

- LH (bimolecular): rate has a maximum vs P_A due to self-inhibition; squared denominator
- ER: rate increases monotonically with P_B (no maximum); linear denominator
- MVK: two-step redox cycle; rate resembles resistances in series
- Mechanism fingerprints (r vs P shape) distinguish models without fitting

> **Session Break (suggested):** Save your work. Part III introduces temperature dependence analysis, building on the rate expressions from Part II.

---
# Part III: Temperature Dependence -- Arrhenius and Eyring Analysis

The temperature dependence of rate constants provides crucial information about reaction energetics and mechanism.

### Arrhenius Equation (Empirical)
$$k = A \exp\left(-\frac{E_a}{RT}\right) \quad \Longrightarrow \quad \ln(k) = \ln(A) - \frac{E_a}{R} \cdot \frac{1}{T}$$

### Eyring Equation (Transition State Theory)
$$k = \frac{k_B T}{h} \exp\left(\frac{\Delta S^\ddagger}{R}\right) \exp\left(-\frac{\Delta H^\ddagger}{RT}\right)$$

Linearized: $\ln\left(\frac{k}{T}\right) = \ln\left(\frac{k_B}{h}\right) + \frac{\Delta S^\ddagger}{R} - \frac{\Delta H^\ddagger}{R} \cdot \frac{1}{T}$

### Connection Between Frameworks
$$E_a = \Delta H^\ddagger + RT \quad \text{and} \quad A = \frac{e \cdot k_B T}{h} \exp\left(\frac{\Delta S^\ddagger}{R}\right)$$

## 3.1 Arrhenius Analysis

We load temperature-dependent rate constant data and extract $E_a$ and $A$.

In [ ]:
# Load experimental data
df_arr = pd.read_csv('data/arrhenius_data_sample.csv', comment='#')
print("Arrhenius Data:")
print(df_arr.to_string(index=False))

T_arr = df_arr['temperature_K'].values
k_arr = df_arr['rate_constant_per_s'].values
k_unc_arr = df_arr['rate_constant_uncertainty'].values

# Arrhenius transformation
inv_T = 1.0 / T_arr
ln_k = np.log(k_arr)
ln_k_unc = k_unc_arr / k_arr  # Error propagation

In [ ]:
# Linear regression: ln(k) = ln(A) - Ea/(R*T)
result_arr = stats.linregress(inv_T, ln_k)

Ea = -result_arr.slope * R / 1000        # kJ/mol
Ea_unc = result_arr.stderr * R / 1000    # kJ/mol
A_pre = np.exp(result_arr.intercept)      # s^-1
r2_arr = result_arr.rvalue**2

print("Arrhenius Analysis Results")
print("=" * 50)
print(f"Ea = {Ea:.1f} +/- {Ea_unc:.1f} kJ/mol")
print(f"A  = {A_pre:.2e} s^-1")
print(f"R^2 = {r2_arr:.6f}")

# Physical reasonableness
log10_A = np.log10(A_pre)
print(f"\nA = 10^{log10_A:.1f} s^-1", end="")
if 10 <= log10_A <= 15:
    print(" (physically reasonable)")
elif log10_A < 10:
    print(" (low -- tight transition state)")
else:
    print(" (high -- loose transition state)")

In [ ]:
# Arrhenius plot with residuals
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

inv_T_fit = np.linspace(inv_T.min() * 0.98, inv_T.max() * 1.02, 200)
ln_k_fit = result_arr.intercept + result_arr.slope * inv_T_fit

# Left: Arrhenius plot
ax1.plot(inv_T_fit * 1000, ln_k_fit, '-',
         color=COLORS['vermillion'], linewidth=2,
         label=f'Fit ($R^2$ = {r2_arr:.4f})')
ax1.errorbar(inv_T * 1000, ln_k, yerr=ln_k_unc,
             fmt='s', markersize=10,
             markerfacecolor=COLORS['blue'],
             markeredgecolor='black', markeredgewidth=1.5,
             ecolor='gray', capsize=4, linestyle='none',
             label='Data')
textstr = (f'$E_a$ = {Ea:.1f} $\\pm$ {Ea_unc:.1f} kJ/mol\n'
           f'$A$ = {A_pre:.2e} s$^{{-1}}$')
ax1.text(0.05, 0.95, textstr, transform=ax1.transAxes, fontsize=11,
         verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
ax1.set_xlabel('1000/$T$ (K$^{-1}$)')
ax1.set_ylabel('ln($k$ / s$^{-1}$)')
ax1.set_title('Arrhenius Plot')
ax1.legend(loc='lower left')
ax1.grid(True, alpha=0.3)

# Right: Residuals
ln_k_pred = result_arr.intercept + result_arr.slope * inv_T
resid_arr = ln_k - ln_k_pred
ax2.plot(inv_T * 1000, resid_arr, 'o', markersize=10,
         markerfacecolor=COLORS['green'],
         markeredgecolor='black', markeredgewidth=1.5)
ax2.axhline(y=0, color='black', linewidth=1)
std_r = resid_arr.std()
ax2.axhline(y=2*std_r, color='gray', linestyle='--', alpha=0.5)
ax2.axhline(y=-2*std_r, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('1000/$T$ (K$^{-1}$)')
ax2.set_ylabel('Residual (ln $k$)')
ax2.set_title('Residual Plot')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3.2 Eyring Analysis

The Eyring equation separates the temperature dependence into enthalpic ($\Delta H^\ddagger$) and entropic ($\Delta S^\ddagger$) contributions.

Plot $\ln(k/T)$ vs $1/T$:
- Slope $= -\Delta H^\ddagger / R$
- Intercept $= \ln(k_B/h) + \Delta S^\ddagger / R = 23.76 + \Delta S^\ddagger / R$

In [ ]:
# Eyring transformation
ln_k_over_T = np.log(k_arr / T_arr)

result_eyr = stats.linregress(inv_T, ln_k_over_T)

dH_ddagger = -result_eyr.slope * R / 1000        # kJ/mol
dH_unc = result_eyr.stderr * R / 1000
dS_ddagger = R * (result_eyr.intercept - ln_kB_over_h)  # J/(mol K)
dS_unc = R * result_eyr.intercept_stderr
r2_eyr = result_eyr.rvalue**2

print("Eyring Analysis Results")
print("=" * 50)
print(f"dH_ddagger = {dH_ddagger:.1f} +/- {dH_unc:.1f} kJ/mol")
print(f"dS_ddagger = {dS_ddagger:.1f} +/- {dS_unc:.1f} J/(mol K)")
print(f"R^2 = {r2_eyr:.6f}")
print()

# Interpret dS_ddagger
if dS_ddagger < -100:
    print(f"dS_ddagger = {dS_ddagger:.0f} J/(mol K) -- very tight "
          f"transition state (associative)")
elif dS_ddagger < 0:
    print(f"dS_ddagger = {dS_ddagger:.0f} J/(mol K) -- ordered "
          f"transition state")
elif dS_ddagger > 50:
    print(f"dS_ddagger = {dS_ddagger:.0f} J/(mol K) -- loose "
          f"transition state (dissociative)")
else:
    print(f"dS_ddagger = {dS_ddagger:.0f} J/(mol K) -- near zero "
          f"(little structural change)")

In [ ]:
# Dual-panel Arrhenius + Eyring comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ln_kT_fit = result_eyr.intercept + result_eyr.slope * inv_T_fit

# Arrhenius
ax1.plot(inv_T_fit * 1000, ln_k_fit, '-',
         color=COLORS['vermillion'], linewidth=2,
         label=f'Fit ($R^2$ = {r2_arr:.4f})')
ax1.errorbar(inv_T * 1000, ln_k, yerr=ln_k_unc,
             fmt='s', markersize=9, markerfacecolor=COLORS['blue'],
             markeredgecolor='black', markeredgewidth=1.5,
             ecolor='gray', capsize=4, linestyle='none', label='Data')
ax1.text(0.05, 0.95,
         f'$E_a$ = {Ea:.1f} kJ/mol\n$A$ = {A_pre:.2e} s$^{{-1}}$',
         transform=ax1.transAxes, fontsize=11,
         verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
ax1.set_xlabel('1000/$T$ (K$^{-1}$)')
ax1.set_ylabel('ln($k$ / s$^{-1}$)')
ax1.set_title('Arrhenius Plot')
ax1.legend(loc='lower left')
ax1.grid(True, alpha=0.3)

# Eyring
ax2.plot(inv_T_fit * 1000, ln_kT_fit, '-',
         color=COLORS['vermillion'], linewidth=2,
         label=f'Fit ($R^2$ = {r2_eyr:.4f})')
ax2.errorbar(inv_T * 1000, ln_k_over_T, yerr=ln_k_unc,
             fmt='s', markersize=9, markerfacecolor=COLORS['orange'],
             markeredgecolor='black', markeredgewidth=1.5,
             ecolor='gray', capsize=4, linestyle='none', label='Data')
ax2.text(0.05, 0.95,
         f'$\\Delta H^\\ddagger$ = {dH_ddagger:.1f} kJ/mol\n'
         f'$\\Delta S^\\ddagger$ = {dS_ddagger:.0f} J/(mol K)',
         transform=ax2.transAxes, fontsize=11,
         verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
ax2.set_xlabel('1000/$T$ (K$^{-1}$)')
ax2.set_ylabel('ln($k/T$ / s$^{-1}$ K$^{-1}$)')
ax2.set_title('Eyring Plot')
ax2.legend(loc='lower left')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Consistency check
T_mean = np.mean(T_arr)
print(f"\nConsistency: Ea = {Ea:.1f} kJ/mol, "
      f"dH + RT = {dH_ddagger + R*T_mean/1000:.1f} kJ/mol "
      f"(diff = {abs(Ea - dH_ddagger - R*T_mean/1000):.1f} kJ/mol)")

---
## 3.3 Apparent vs. Intrinsic Activation Energy

For LH kinetics $r = k \cdot \theta_A$, the **apparent** $E_a$ depends on the coverage regime:

| Coverage Regime | $E_{app}$ |
|----------------|-----------|
| Low ($KP \ll 1$) | $E_a + \Delta H_{ads}$ |
| High ($KP \gg 1$) | $E_a$ (intrinsic) |

Since $\Delta H_{ads} < 0$ (exothermic), $E_{app}$ is **lower** at low coverage.

In [ ]:
def lh_rate_unimolecular_T(T, P_A, A_k, Ea_intrinsic, K0, dH_ads):
    """
    Temperature-dependent unimolecular LH rate.

    Combines Arrhenius k(T) with van 't Hoff K(T).
    """
    T = np.asarray(T, dtype=float)
    k_surface = A_k * np.exp(-Ea_intrinsic / (R * T))
    K_ads = K0 * np.exp(-dH_ads / (R * T))
    theta_A = K_ads * P_A / (1 + K_ads * P_A)
    return k_surface * theta_A


# ADJUSTABLE PARAMETERS
Ea_intr = 85e3      # Intrinsic Ea = 85 kJ/mol
dH_ads = -40e3      # Adsorption enthalpy = -40 kJ/mol
A_k_val = 1e13      # Pre-exponential for k (s^-1)
K0_val = 1e-5       # Pre-exponential for K (bar^-1)

T_range_ea = np.linspace(350, 700, 500)
P_low = 0.01        # bar (low coverage)
P_high = 100.0      # bar (high coverage)

rate_low_P = lh_rate_unimolecular_T(T_range_ea, P_low,
                                     A_k_val, Ea_intr, K0_val, dH_ads)
rate_high_P = lh_rate_unimolecular_T(T_range_ea, P_high,
                                      A_k_val, Ea_intr, K0_val, dH_ads)

Ea_app_low = (Ea_intr + dH_ads) / 1000    # kJ/mol
Ea_app_high = Ea_intr / 1000              # kJ/mol

print(f"Expected E_app:")
print(f"  Low coverage  (P = {P_low} bar): {Ea_app_low:.0f} kJ/mol "
      f"(= Ea + dH_ads)")
print(f"  High coverage (P = {P_high} bar): {Ea_app_high:.0f} kJ/mol "
      f"(= intrinsic Ea)")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: Arrhenius plots at both pressures
ax1.plot(1000 / T_range_ea, np.log(rate_low_P),
         color=COLORS['blue'], linewidth=2.5,
         label=f'$P_A$ = {P_low} bar (low coverage)')
ax1.plot(1000 / T_range_ea, np.log(rate_high_P),
         color=COLORS['vermillion'], linewidth=2.5, linestyle='--',
         label=f'$P_A$ = {P_high} bar (high coverage)')
ax1.text(0.55, 0.30,
         f'Low $P$: $E_{{app}}$ = {Ea_app_low:.0f} kJ/mol\n'
         f'($E_a + \\Delta H_{{ads}}$)',
         transform=ax1.transAxes, fontsize=10,
         color=COLORS['blue'],
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax1.text(0.55, 0.12,
         f'High $P$: $E_{{app}}$ = {Ea_app_high:.0f} kJ/mol\n'
         f'(intrinsic $E_a$)',
         transform=ax1.transAxes, fontsize=10,
         color=COLORS['vermillion'],
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax1.set_xlabel('1000/$T$ (K$^{-1}$)')
ax1.set_ylabel('ln($r$)')
ax1.set_title('Apparent vs. Intrinsic Activation Energy')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Right: Local E_app vs T
inv_T_ea = 1.0 / T_range_ea
Ea_local_low = -R * np.gradient(np.log(rate_low_P),
                                 inv_T_ea) / 1000
Ea_local_high = -R * np.gradient(np.log(rate_high_P),
                                  inv_T_ea) / 1000
ax2.plot(T_range_ea, Ea_local_low, color=COLORS['blue'],
         linewidth=2.5, label=f'$P_A$ = {P_low} bar')
ax2.plot(T_range_ea, Ea_local_high, color=COLORS['vermillion'],
         linewidth=2.5, linestyle='--',
         label=f'$P_A$ = {P_high} bar')
ax2.axhline(y=Ea_intr/1000, color='gray', linestyle=':',
            alpha=0.7, label=f'$E_a$ = {Ea_intr/1000:.0f} kJ/mol')
ax2.axhline(y=(Ea_intr+dH_ads)/1000, color='gray', linestyle='--',
            alpha=0.5,
            label=f'$E_a + \\Delta H_{{ads}}$ = '
                  f'{(Ea_intr+dH_ads)/1000:.0f} kJ/mol')
ax2.set_xlabel('Temperature (K)')
ax2.set_ylabel('$E_{app}$ (kJ/mol)')
ax2.set_title('Apparent Activation Energy vs. Temperature')
ax2.legend(loc='center right', fontsize=9)
ax2.set_ylim(30, 100)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3.4 Compensation Effect

The enthalpy-entropy compensation (isokinetic) effect: reactions with higher $\Delta H^\ddagger$ tend to have more positive $\Delta S^\ddagger$. Equivalently, $\ln(A)$ and $E_a$ are linearly correlated:

$$\ln(A) = \frac{E_a}{RT_{iso}} + C$$

where $T_{iso}$ is the isokinetic temperature at which all catalysts in the series give the same rate constant.

In [ ]:
# Simulated compensation effect: 6 catalysts for the same reaction
np.random.seed(42)
T_iso = 500  # Isokinetic temperature (K)
k_iso = 0.1  # Rate constant at T_iso (s^-1)

Ea_catalysts = np.array([50, 65, 80, 95, 110, 125]) * 1000  # J/mol
ln_A_catalysts = np.log(k_iso) + Ea_catalysts / (R * T_iso)
ln_A_catalysts += np.random.normal(0, 0.5, len(Ea_catalysts))
A_catalysts = np.exp(ln_A_catalysts)
catalyst_names = ['Pt/Al2O3', 'Pd/Al2O3', 'Rh/Al2O3',
                  'Pt/SiO2', 'Pd/SiO2', 'Rh/SiO2']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: ln(A) vs Ea
Ea_kJ_cat = Ea_catalysts / 1000
result_comp = stats.linregress(Ea_kJ_cat, ln_A_catalysts)
Ea_fit_r = np.linspace(Ea_kJ_cat.min() - 5, Ea_kJ_cat.max() + 5, 100)
T_iso_fit = 1000 / (result_comp.slope * R)

ax1.plot(Ea_fit_r,
         result_comp.intercept + result_comp.slope * Ea_fit_r,
         '-', color=COLORS['vermillion'], linewidth=2,
         label=f'Fit: $T_{{iso}}$ = {T_iso_fit:.0f} K')
for i, name in enumerate(catalyst_names):
    ax1.plot(Ea_kJ_cat[i], ln_A_catalysts[i], 'o', markersize=12,
             markerfacecolor=COLORS['blue'],
             markeredgecolor='black', markeredgewidth=1.5)
    ax1.annotate(name, xy=(Ea_kJ_cat[i], ln_A_catalysts[i]),
                 xytext=(5, 5), textcoords='offset points', fontsize=9)
ax1.set_xlabel('$E_a$ (kJ/mol)')
ax1.set_ylabel('ln($A$ / s$^{-1}$)')
ax1.set_title('Compensation Effect')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# Right: Arrhenius curves converging at T_iso
T_plot_comp = np.linspace(350, 700, 300)
colors_cat = plt.cm.viridis(np.linspace(0.1, 0.9, len(catalyst_names)))
for i, (name, ea, a, clr) in enumerate(zip(catalyst_names,
                                             Ea_catalysts,
                                             A_catalysts,
                                             colors_cat)):
    k_vals = a * np.exp(-ea / (R * T_plot_comp))
    ax2.semilogy(T_plot_comp, k_vals, color=clr, linewidth=1.5,
                 label=f'{name} ($E_a$={ea/1000:.0f})')
ax2.axvline(x=T_iso, color='black', linestyle='--', linewidth=1.5,
            alpha=0.7)
ax2.text(T_iso + 5, 1e-4, f'$T_{{iso}}$ = {T_iso} K', fontsize=11)
ax2.set_xlabel('Temperature (K)')
ax2.set_ylabel('Rate Constant, $k$ (s$^{-1}$)')
ax2.set_title('Isokinetic Relationship')
ax2.legend(loc='upper left', fontsize=8)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f"At T_iso, all catalysts give approximately the same k.")
print(f"Above T_iso: high-Ea catalysts are faster.")
print(f"Below T_iso: low-Ea catalysts are faster.")

---
## 3.5 Complete Arrhenius + Eyring Workflow

We wrap the analysis into a reusable function that produces Arrhenius and Eyring parameters with uncertainties, consistency checks, and diagnostic plots.

In [ ]:
def full_temperature_analysis(T, k, k_uncertainty=None,
                               plot=True, verbose=True):
    """
    Complete Arrhenius + Eyring analysis on rate constant data.

    Parameters
    ----------
    T : array-like
        Temperature (K)
    k : array-like
        Rate constant values
    k_uncertainty : array-like, optional
        Uncertainties in k
    plot : bool
        Generate diagnostic plots
    verbose : bool
        Print results

    Returns
    -------
    dict
        Ea_kJ_mol, A, dH_kJ_mol, dS_J_mol_K, R_squared_arr,
        R_squared_eyr, and their uncertainties
    """
    R_gas = 8.314
    kB_val = 1.380649e-23
    h_val = 6.62607e-34
    ln_kB_h = np.log(kB_val / h_val)

    T = np.asarray(T, dtype=float)
    k = np.asarray(k, dtype=float)

    inv_T_v = 1.0 / T
    ln_k_v = np.log(k)
    ln_kT_v = np.log(k / T)

    # Arrhenius
    res_a = stats.linregress(inv_T_v, ln_k_v)
    Ea_v = -res_a.slope * R_gas / 1000
    Ea_e = res_a.stderr * R_gas / 1000
    A_v = np.exp(res_a.intercept)
    A_e = A_v * res_a.intercept_stderr
    r2a = res_a.rvalue**2
    resid_a = ln_k_v - (res_a.intercept + res_a.slope * inv_T_v)

    # Eyring
    res_e = stats.linregress(inv_T_v, ln_kT_v)
    dH_v = -res_e.slope * R_gas / 1000
    dH_e = res_e.stderr * R_gas / 1000
    dS_v = R_gas * (res_e.intercept - ln_kB_h)
    dS_e = R_gas * res_e.intercept_stderr
    r2e = res_e.rvalue**2
    resid_e = ln_kT_v - (res_e.intercept + res_e.slope * inv_T_v)

    if verbose:
        print("Complete Temperature Analysis")
        print("=" * 55)
        print(f"Arrhenius: Ea = {Ea_v:.1f} +/- {Ea_e:.1f} kJ/mol, "
              f"A = {A_v:.2e}, R^2 = {r2a:.6f}")
        print(f"Eyring:    dH = {dH_v:.1f} +/- {dH_e:.1f} kJ/mol, "
              f"dS = {dS_v:.1f} +/- {dS_e:.1f} J/(mol K), "
              f"R^2 = {r2e:.6f}")
        T_avg = np.mean(T)
        print(f"Check:     Ea - dH = {Ea_v - dH_v:.1f} kJ/mol "
              f"(expected RT = {R_gas*T_avg/1000:.1f} kJ/mol)")

    if plot:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        inv_T_sm = np.linspace(inv_T_v.min() * 0.98,
                                inv_T_v.max() * 1.02, 200)
        ln_k_err = (k_uncertainty / k
                    if k_uncertainty is not None else None)

        # Top-left: Arrhenius
        ax = axes[0, 0]
        ax.plot(inv_T_sm * 1000,
                res_a.intercept + res_a.slope * inv_T_sm,
                '-', color='#D55E00', linewidth=2,
                label=f'Fit ($R^2$ = {r2a:.4f})')
        if ln_k_err is not None:
            ax.errorbar(inv_T_v * 1000, ln_k_v, yerr=ln_k_err,
                        fmt='s', markersize=8,
                        markerfacecolor='#0072B2',
                        markeredgecolor='black', markeredgewidth=1,
                        ecolor='gray', capsize=3, linestyle='none',
                        label='Data')
        else:
            ax.plot(inv_T_v * 1000, ln_k_v, 's', markersize=8,
                    markerfacecolor='#0072B2',
                    markeredgecolor='black', markeredgewidth=1,
                    label='Data')
        ax.text(0.05, 0.95,
                f'$E_a$ = {Ea_v:.1f} kJ/mol\n$A$ = {A_v:.2e}',
                transform=ax.transAxes, fontsize=10,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat',
                          alpha=0.8))
        ax.set_xlabel('1000/$T$ (K$^{-1}$)')
        ax.set_ylabel('ln($k$)')
        ax.set_title('Arrhenius Plot')
        ax.legend(loc='lower left')
        ax.grid(True, alpha=0.3)

        # Top-right: Eyring
        ax = axes[0, 1]
        ax.plot(inv_T_sm * 1000,
                res_e.intercept + res_e.slope * inv_T_sm,
                '-', color='#D55E00', linewidth=2,
                label=f'Fit ($R^2$ = {r2e:.4f})')
        if ln_k_err is not None:
            ax.errorbar(inv_T_v * 1000, ln_kT_v, yerr=ln_k_err,
                        fmt='s', markersize=8,
                        markerfacecolor='#E69F00',
                        markeredgecolor='black', markeredgewidth=1,
                        ecolor='gray', capsize=3, linestyle='none',
                        label='Data')
        else:
            ax.plot(inv_T_v * 1000, ln_kT_v, 's', markersize=8,
                    markerfacecolor='#E69F00',
                    markeredgecolor='black', markeredgewidth=1,
                    label='Data')
        ax.text(0.05, 0.95,
                f'$\\Delta H^\\ddagger$ = {dH_v:.1f} kJ/mol\n'
                f'$\\Delta S^\\ddagger$ = {dS_v:.0f} J/(mol K)',
                transform=ax.transAxes, fontsize=10,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat',
                          alpha=0.8))
        ax.set_xlabel('1000/$T$ (K$^{-1}$)')
        ax.set_ylabel('ln($k/T$)')
        ax.set_title('Eyring Plot')
        ax.legend(loc='lower left')
        ax.grid(True, alpha=0.3)

        # Bottom-left: Arrhenius residuals
        ax = axes[1, 0]
        ax.plot(inv_T_v * 1000, resid_a, 'o', markersize=10,
                markerfacecolor='#009E73',
                markeredgecolor='black', markeredgewidth=1.5)
        ax.axhline(y=0, color='black', linewidth=1)
        s = resid_a.std()
        ax.axhline(y=2*s, color='gray', linestyle='--', alpha=0.5)
        ax.axhline(y=-2*s, color='gray', linestyle='--', alpha=0.5)
        ax.set_xlabel('1000/$T$ (K$^{-1}$)')
        ax.set_ylabel('Residual (ln $k$)')
        ax.set_title('Arrhenius Residuals')
        ax.grid(True, alpha=0.3)

        # Bottom-right: Eyring residuals
        ax = axes[1, 1]
        ax.plot(inv_T_v * 1000, resid_e, 'o', markersize=10,
                markerfacecolor='#009E73',
                markeredgecolor='black', markeredgewidth=1.5)
        ax.axhline(y=0, color='black', linewidth=1)
        s = resid_e.std()
        ax.axhline(y=2*s, color='gray', linestyle='--', alpha=0.5)
        ax.axhline(y=-2*s, color='gray', linestyle='--', alpha=0.5)
        ax.set_xlabel('1000/$T$ (K$^{-1}$)')
        ax.set_ylabel('Residual (ln $k/T$)')
        ax.set_title('Eyring Residuals')
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

    return {
        'Ea_kJ_mol': Ea_v, 'Ea_uncertainty': Ea_e,
        'A': A_v, 'A_uncertainty': A_e,
        'dH_kJ_mol': dH_v, 'dH_uncertainty': dH_e,
        'dS_J_mol_K': dS_v, 'dS_uncertainty': dS_e,
        'R_squared_arr': r2a, 'R_squared_eyr': r2e
    }


# Demonstrate workflow
results = full_temperature_analysis(T_arr, k_arr,
                                     k_uncertainty=k_unc_arr)

---
# Exercises

The following exercises span all three parts of this notebook.

## Exercise 1: Langmuir Sensitivity Analysis (Part I)

**Task:**
1. In the Henry's law regime ($KP \ll 1$), show mathematically that $\theta \approx KP$
2. Assess how a 10% uncertainty in $K$ affects coverage at:
   - $P = 0.1/K$ (Henry regime)
   - $P = 1/K$ (half-coverage)
   - $P = 10/K$ (near saturation)
3. At which regime is the coverage most sensitive to uncertainty in $K$?

In [ ]:
# YOUR CODE HERE
# 1. Taylor expand theta = KP/(1+KP) for KP << 1
# 2. Calculate theta for K and K*(1+/-0.10) at three pressures
# 3. Report percent change in theta for each case

pass  # Replace with your implementation

## Exercise 2: Fit the ER Model to CO Oxidation Data (Part II)

**Task:**
1. Define an `er_model_1var` function: $r = c P_A / (1 + d P_A)$ (denominator exponent 1, not 2)
2. Fit it to the 400 K data using `curve_fit`
3. Calculate $R^2$ and plot residuals alongside the LH residuals
4. Which model better captures the data, and why?

In [ ]:
def er_model_1var(P_A, c, d):
    """
    Simplified ER model for fitting at fixed P_B.

    r = c * P_A / (1 + d * P_A)

    Parameters
    ----------
    P_A : float or array-like
        Partial pressure (bar)
    c : float
        Lumped kinetic-adsorption parameter
    d : float
        Effective adsorption parameter (bar^-1)

    Returns
    -------
    float or ndarray
        Reaction rate
    """
    pass  # Replace with your implementation


# YOUR CODE HERE
# 1. Fit er_model_1var to P_CO_bar and rate_data
# 2. Calculate R^2
# 3. Plot residuals: ER vs LH
# 4. Discuss which model fits better

pass  # Replace with your implementation

## Exercise 3: LH Rate Maximum Derivation (Part II)

**Task:**
1. Take the derivative $\partial r / \partial P_A$ and set it to zero
2. Show that the maximum occurs at $P_{A,max} = (1 + K_B P_B) / K_A$
3. Verify numerically with $k=1$, $K_A=2$, $K_B=1$, $P_B=0.2$ bar
4. What is $\theta_A$ at the rate maximum?

In [ ]:
# YOUR CODE HERE
# 1. Calculate P_A_max analytically
# 2. Find numerical maximum from a dense array
# 3. Calculate theta_A at the maximum
# 4. What fraction of sites does A occupy at the optimum?

pass  # Replace with your implementation

## Exercise 4: Sensitivity to Outliers in Arrhenius Analysis (Part III)

**Task:**
1. Remove the highest-temperature data point from the Arrhenius dataset
2. Re-run `full_temperature_analysis` on the trimmed data
3. Compare $E_a$ and $\Delta H^\ddagger$ with the full-dataset values
4. Why might the extreme-temperature point have outsized influence on the fit?

In [ ]:
# YOUR CODE HERE
# 1. Create T_trimmed, k_trimmed by excluding the last point
# 2. Run full_temperature_analysis(T_trimmed, k_trimmed)
# 3. Compare with full-dataset results
# 4. Discuss leverage of extreme points

pass  # Replace with your implementation

## Exercise 5: Detecting Non-Arrhenius Behavior (Part III)

**Task:**
1. Generate synthetic data with **two different** $E_a$ values (simulating a mechanism change):
   - Low T (350--450 K): $E_a = 60$ kJ/mol, $A = 10^{10}$ s$^{-1}$
   - High T (450--600 K): $E_a = 120$ kJ/mol, $A = 10^{18}$ s$^{-1}$
2. Fit a single Arrhenius line to the combined data
3. Examine the residuals -- the curved pattern is the diagnostic for non-Arrhenius behavior

In [ ]:
# YOUR CODE HERE
# 1. Generate T_low, T_high, k_low, k_high
# 2. Combine into T_all, k_all
# 3. Fit single Arrhenius line
# 4. Plot residuals -- look for curvature

pass  # Replace with your implementation

## Exercise 6: Competitive Adsorption Design Problem (Part I)

A catalytic reaction uses reactant A with $K_A = 5$ bar$^{-1}$ and $P_A = 0.5$ bar. The feed contains an impurity I with $K_I = 500$ bar$^{-1}$.

**Task:**
1. What is the maximum allowable $P_I$ to maintain at least 90% of the unpoisoned coverage?
2. If $P_I$ cannot be reduced below 0.01 bar, what minimum $P_A$ is needed to achieve $\theta_A = 0.5$?

In [ ]:
# YOUR CODE HERE
# 1. Solve for P_I: theta_A >= 0.9 * theta_A_pure
# 2. Solve for P_A: theta_A = 0.5 with P_I = 0.01 bar

pass  # Replace with your implementation

---
# Reflection Questions

### Part I: Adsorption
1. The Langmuir model assumes all adsorption sites are equivalent. How would the isotherm change if some sites were more energetically favorable than others?
2. Why is adsorption typically exothermic ($\Delta H < 0$)? What does this imply for the stability of adsorbed species at high temperature?

### Part II: Surface Kinetics
3. At low pressures, both LH and ER reduce to the same power-law form ($r \propto P_A$). Why is it essential to collect data at higher pressures for mechanism discrimination?
4. A colleague fits a power-law model and obtains $n = 0.6$. What does a fractional apparent order suggest about the underlying mechanism?
5. In the bimolecular LH mechanism, why does the denominator appear with exponent 2 rather than 1?

### Part III: Temperature Dependence
6. Why do we plot $\ln(k/T)$ vs $1/T$ for Eyring rather than $\ln(k)$ vs $1/T$? What additional information does Eyring provide?
7. A colleague reports $A = 10^{3}$ s$^{-1}$ for a surface reaction. Using the Eyring framework, what does this imply about $\Delta S^\ddagger$? Is this physically reasonable?
8. Explain why an Arrhenius plot might show **downward curvature** at high temperatures for a catalytic reaction.

---
# Summary

## Part I: Adsorption Fundamentals
- Implemented the **Langmuir isotherm** and explored how $K$ affects coverage
- Created **coverage heatmaps** showing combined $T$ and $P$ effects
- Modeled **competitive adsorption** and catalyst poisoning
- Analyzed **TPD spectra** using the Redhead equation

## Part II: Surface Kinetics
- Implemented **three rate expressions** (LH, ER, MVK) and their fingerprints
- Identified the key diagnostic: does $r(P_A)$ show a maximum?
- Fit models to data using `curve_fit` with residual analysis
- Checked physical reasonableness of fitted parameters

## Part III: Temperature Dependence
- Performed **Arrhenius** and **Eyring** analysis with uncertainty propagation
- Demonstrated **apparent vs. intrinsic** $E_a$ for LH kinetics
- Explored the **compensation effect** across catalyst families
- Built a reusable `full_temperature_analysis` workflow function

### Key Connections
- **Part I to Part II**: Coverage ($\theta$) from the Langmuir isotherm enters directly into LH/ER rate expressions
- **Part II to Part III**: Temperature dependence of rate constants ($k$) and adsorption constants ($K$) creates coverage-dependent apparent activation energies
- **All parts**: Residual analysis is the universal tool for model validation

**Next notebook:** NB3 covers ideal reactor design and catalyst characterization (Chapter 3).